In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchainhub import Client

C:\Users\kirit\AppData\Local\Temp\ipykernel_31728\1295230256.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
from langgraph.prebuilt import create_react_agent  #reasoning and action = react 

In [20]:
#load env variables
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\kirit\AppData\Local\Temp\ipykernel_31728\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [21]:


@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )



In [5]:
result=search_tool.invoke("What is the capital of France?")
result

[{'title': 'Capital of France - Simple English Wikipedia, the free encyclopedia',
  'url': 'https://simple.wikipedia.org/wiki/Capital_of_France',
  'content': 'Wikipedia\nThe Free Encyclopedia\n\n## Contents\n\n# Capital of France\n\nThe capital of France is Paris. In the course of history, the national capital has been in many locations other than Paris.\n\n## History\n\n### List of capitals of France\n\n## References\n\nWikimedia Foundation\nPowered by MediaWiki',
  'score': 0.93735445},
 {'title': 'What is the Capital of France? | Mappr',
  'url': 'https://www.mappr.co/capital-cities/france',
  'content': 'Skip to content\n\nmappr logo\n\nmappr logo\n\n# What is the Capital of France?\n\nParis is the capital of France. Also known as the “City of Light,” is not only the capital but also the most populous city of France. Paris is renowned globally for its art, fashion, gastronomy, and culture.\n\n📌 Quick Facts: Paris\n\n📍 Capital of\n\n👥 City population\n\n≈ 2.1 million\n\n🏠 Median 3-

In [6]:
#llm

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [7]:
result=llm.invoke("What year is it?")
result

d:\conda_envs\aii_agents\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'It is 2024.', 'extras': {'signature': 'EmAKXgFpFH0TY+STt7X2d4cuR1CeY//F6j09hZJGWMyLcMGCds2xnL/TkIVdhD0lbkEP9RvKOCGliiZo0bY6RiS7K+hBIK1zrSG5v4P8MgBeViXwRTPbg4FlE7O1NCmN0Tc='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1927c-e9bb-7793-8b80-b09f98f3140b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 8, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}})

In [8]:
#prompt
#recommeded prompt for create react agent
client = Client()
prompt = client.pull("hwchase17/react")

C:\Users\kirit\AppData\Local\Temp\ipykernel_31728\1736375509.py:4: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("hwchase17/react")
d:\conda_envs\aii_agents\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


In [9]:
prompt

'{"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "Answer the following questions as best you can. You have access to the following tools:\\n\\n{tools}\\n\\nUse the following format:\\n\\nQuestion: the input question you must answer\\nThought: you should always think about what to do\\nAction: the action to take, should be one of [{tool_names}]\\nAction Input: the input to the action\\nObservation: the result of the action\\n... (this Thought/Action/Action Input/Observation can repeat N times)\\nThought: I now know the final answer\\nFinal Answer: the final answer to the original input question\\n\\nBegin!\\n\\nQuestion: {input}\\nThought:{agent_scratchpad}", "input_variables": ["agent_scratchpad", "input", "tool_names", "tools"], "template_format": "f-string", "partial_variables": {}}}'

In [22]:
tools= [search_tool,get_weather_data]

In [23]:
#create agent

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt
)

C:\Users\kirit\AppData\Local\Temp\ipykernel_31728\2055112834.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [24]:
response = agent.invoke({
    "messages":("user", "Find the capital of Nepal and its current weather")
})

d:\conda_envs\aii_agents\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\conda_envs\aii_agents\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\conda_envs\aii_agents\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [25]:
response

{'messages': [HumanMessage(content='Find the capital of Nepal and its current weather', additional_kwargs={}, response_metadata={}, id='3a8c8c84-9ed2-4821-b1c4-a29726abb4c9'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "capital of Nepal"}'}, '__gemini_function_call_thought_signatures__': {'call_773937': 'EmAKXgFpFH0TehQ98N3SM0ZK6isskFk9wDio5do8aKzpq9F/4wFxXhzor7pDGEx+bb9G0NStdPwfFMgUNqb/7fHcJCeDvSKLRbnWbA/JqeW1eGRKl2/WmcSqcZ4cGhj7YNU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a192fc-4e0d-7fb0-8950-0a970e6f664b-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'capital of Nepal'}, 'id': 'call_773937', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 370, 'output_tokens': 1, 'total_tokens': 371, 'input_token_details': {'cache_read': 0}}),
  Too

In [42]:
result=response['messages'][-1].content

In [44]:
type(result)

list

In [45]:
result

[{'type': 'text',
  'text': 'The capital of Nepal is **Kathmandu**. \n\nThe current weather in Kathmandu is **13°C and Partly Cloudy** with a humidity of 93%.',
  'extras': {'signature': 'EmAKXgFpFH0TX/g0w4SBFzQGjGdLnv615IiZ/TOYFrEbNtOBIkIXAMiIcWy8Y0hPepRHkNeUesGTuVTL7jyEhrsqzb7+rqdxYUsbW0QX7RThK84+7DV44TEqbJ+pMzLRp6I='}}]

In [47]:
result[0]['text']

'The capital of Nepal is **Kathmandu**. \n\nThe current weather in Kathmandu is **13°C and Partly Cloudy** with a humidity of 93%.'